# Phase 3 -- Entity Feature / Representation Building

Alert Intelligence Engine -- master plan Phase 3 (section 20): "Representation + historical context." Gate: feature audit passes.

Covers the two sheets that share the entity/watchlist-relationship problem: `CustomerViolation` (Customer Name Alerts) and `TransactionNameViolation` (Transaction Name Alerts). `Rule` (Transaction Intelligence) is Phase 4, not this notebook.

**No model training happens in this phase.** This builds *representations* only -- the anomaly/novelty model comes in Phase 5. No hand-written name/DOB/nationality match weight exists anywhere here (master plan non-negotiable rule); the character-n-gram TF-IDF and one-hot encoders below are all *learned, generic* representations, not decision logic.

Feature families built here (master plan section 5):
- Character n-gram TF-IDF for `Alerted Party Name` and `Hit Details (Name)` -- the "strong, explainable, local, non-LLM baseline" the master plan calls for
- Context categorical encoding (Customer Type, Alerted Party role, Alert Type, Screening List, Branch)
- DOB representation + explicit missingness (handles the Phase 2 finding that `Hit Details (DOB)` needs year-only fallback)
- Nationality representation (built on Phase 2's normalized country values)
- Historical customer context -- prior-alert count, provably non-leaking (strictly-prior-only)
- `Matched Screening %` as a plain numeric feature -- explicitly sanctioned as a *feature*, never a hardcoded decision threshold

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

## 1. Load Phase 2 normalized data

In [2]:
from pipelines.normalization.pipeline import run_phase2_pipeline

normalized_sheets, phase2_report = run_phase2_pipeline(REPO_ROOT, persist=False)
assert phase2_report["overall_status"] == "PASS"

entity_sheets = {k: v for k, v in normalized_sheets.items() if k != "Rule"}
for name, df in entity_sheets.items():
    print(f"{name}: {len(df)} rows, {len(df.columns)} columns")

CustomerViolation: 2397 rows, 54 columns
TransactionNameViolation: 2000 rows, 60 columns


## 2. Character n-gram name representation

Fit on `CustomerViolation`'s `Alerted Party Name (Normalized)` as an example. char_wb n-grams (2-4 characters) -- robust to small spelling variation without any hand-written similarity threshold; the model (Phase 5) learns what n-gram patterns matter, not a person.

In [3]:
from features.entity_features import fit_name_vectorizer, transform_names

cv_df = entity_sheets["CustomerViolation"]
name_vec = fit_name_vectorizer(cv_df["Alerted Party Name (Normalized)"])
name_matrix = transform_names(name_vec, cv_df["Alerted Party Name (Normalized)"])

print(f"Vocabulary size: {len(name_vec.vocabulary_)}")
print(f"Matrix shape: {name_matrix.shape}, density: {name_matrix.nnz / (name_matrix.shape[0]*name_matrix.shape[1]):.4%}")
print("\nExample n-grams learned:", list(name_vec.vocabulary_.keys())[:15])

missing_name_rows = cv_df["Alerted Party Name (Normalized)"].isna().sum()
print(f"\nRows with missing name: {missing_name_rows} -- these get an all-zero vector, "
      f"never a fabricated name signal.")

Vocabulary size: 4296
Matrix shape: (2397, 4296), density: 1.0119%

Example n-grams learned: [' T', 'TA', 'AR', 'RE', 'EK', 'K ', ' TA', 'TAR', 'ARE', 'REK', 'EK ', ' TAR', ' M', 'MO', 'OR']

Rows with missing name: 0 -- these get an all-zero vector, never a fabricated name signal.


## 3. `Hit Details (DOB)` handling -- year-only fallback in action

Phase 2 classified this field instead of guessing (see `notebooks/02_data_pipeline.ipynb` section 2). Here, DOB features fall back to a year-only age estimate when a full date wasn't resolvable, and leave the feature NaN (imputed to 0 only at final matrix assembly, with a separate `hit_dob_resolved` flag) when nothing could be safely resolved at all.

In [4]:
from features.entity_features import build_dob_features

dob_features = build_dob_features(cv_df, "CustomerViolation")
print(dob_features.describe())
print()
print(f"Hit DOB resolved (full date or year): {dob_features['hit_dob_resolved'].sum()}/{len(dob_features)}")
print(f"Hit DOB unresolved (flagged, not guessed): {dob_features['hit_dob_unresolved'].sum()}/{len(dob_features)}")

       alerted_party_age_years  alerted_party_dob_missing  hit_age_years  hit_dob_resolved  hit_dob_multi_value  hit_dob_unresolved
count              2353.000000                2397.000000    1236.000000       2397.000000          2397.000000         2397.000000
mean                 34.367281                   0.018356      53.549851          0.515645             0.177722            0.004172
std                  15.643370                   0.134264      15.885009          0.499859             0.382358            0.064469
min                   0.095825                   0.000000       0.276523          0.000000             0.000000            0.000000
25%                  25.965777                   0.000000      41.455852          0.000000             0.000000            0.000000
50%                  34.395619                   0.000000      55.000000          1.000000             0.000000            0.000000
75%                  42.592745                   0.000000      65.165640    

## 4. Historical customer context -- leakage check

Prior-alert count per customer, using only alerts strictly before the current one's `Alert Generated Date & Time`. Demonstrated on a real repeat customer from the data.

**Known limitation, stated plainly:** several alerts in this workbook share the exact same second-level timestamp for the same customer (batch-generated). There is no information in the source data to causally order same-timestamp alerts, so ties are broken by original row order (stable sort) -- not a claim of true causal ordering. The display below must use the same stable sort the function itself uses (`kind="mergesort"`), or tied rows can appear out of order purely from an unrelated, unstable secondary sort -- that's a display artifact, not a leakage bug, and it is worth knowing about before trusting any downstream chart that resorts this column.

In [5]:
from features.entity_features import build_historical_customer_context

history = build_historical_customer_context(cv_df, "CustomerViolation")
cv_with_history = cv_df.copy()
cv_with_history["prior_alert_count"] = history["customer_prior_alert_count"]

# find a customer with several alerts and show the count only ever goes up over time.
# kind="mergesort" (stable) matches what build_historical_customer_context uses
# internally -- an unstable sort here would reorder same-timestamp ties
# differently and falsely look like a leakage bug.
repeat_customer = cv_with_history["customer_id"].value_counts().index[0]
example = cv_with_history[cv_with_history["customer_id"] == repeat_customer][
    ["customer_id", "Alert Generated Date & Time (Parsed)", "prior_alert_count"]
].sort_values("Alert Generated Date & Time (Parsed)", kind="mergesort")
print(example.to_string(index=False))

n_tied_timestamps = example["Alert Generated Date & Time (Parsed)"].duplicated().sum()
print(f"\n{n_tied_timestamps} rows share a timestamp with another row for this customer "
      f"(tie-broken by original row order, not causal order).")

diffs = example["prior_alert_count"].diff().dropna()
assert (diffs >= 0).all(), "prior_alert_count must never decrease within a customer's own timeline"
print("Confirmed: non-decreasing over time for this customer -- no future alert is counted as prior.")

                       customer_id Alert Generated Date & Time (Parsed)  prior_alert_count
customerviolation:customer:2503924                  2026-06-01 17:10:00                0.0
customerviolation:customer:2503924                  2026-06-01 17:10:00                1.0
customerviolation:customer:2503924                  2026-06-01 17:10:00                2.0
customerviolation:customer:2503924                  2026-06-01 17:10:00                3.0
customerviolation:customer:2503924                  2026-06-01 17:10:00                4.0
customerviolation:customer:2503924                  2026-06-01 17:10:00                5.0
customerviolation:customer:2503924                  2026-06-01 17:10:01                6.0
customerviolation:customer:2503924                  2026-06-01 17:10:01                7.0
customerviolation:customer:2503924                  2026-06-01 17:10:01                8.0
customerviolation:customer:2503924                  2026-06-01 17:10:01                9.0

## 5. Full entity feature matrix -- both sheets

In [6]:
from features.entity_features import build_entity_features

artifacts_by_sheet = {}
matrices = {}

for sheet_name, df in entity_sheets.items():
    matrix, block_names, artifacts = build_entity_features(df, sheet_name)
    matrices[sheet_name] = matrix
    artifacts_by_sheet[sheet_name] = artifacts
    print(f"=== {sheet_name} ===")
    print(f"  rows={matrix.shape[0]}, features={matrix.shape[1]}, nnz={matrix.nnz}")
    print(f"  feature blocks: {block_names}")
    dense_check = matrix.toarray()
    assert not np.isnan(dense_check).any(), "no NaN permitted in the final matrix"
    assert not np.isinf(dense_check).any(), "no inf permitted in the final matrix"
    print("  PASS: no NaN/inf in final matrix")
    print()

=== CustomerViolation ===
  rows=2397, features=8532, nnz=226693
  feature blocks: ['name_tfidf::Alerted Party Name (Normalized)', 'name_tfidf::Hit Details (Name) (Normalized)', 'categorical::Customer Type', 'categorical::Alerted Party', 'categorical::Alert Type', 'categorical::Sanctions Screening List Name', 'categorical::Branch Name', 'nationality::Alerted Party Nationality (Normalized)', 'nationality::Hit Details (Nationality) (Normalized)', 'numeric::Matched Screening % (Parsed),alerted_party_age_years,alerted_party_dob_missing,hit_age_years,hit_dob_resolved,hit_dob_multi_value,hit_dob_unresolved,customer_prior_alert_count']


  PASS: no NaN/inf in final matrix

=== TransactionNameViolation ===
  rows=2000, features=5654, nnz=151712
  feature blocks: ['name_tfidf::Alerted Party Name (Normalized)', 'name_tfidf::Hit Details (Name) (Normalized)', 'categorical::Customer Type', 'categorical::Alerted Party', 'categorical::Alert Type', 'categorical::Sanctions Screening List Name', 'categorical::Branch Name', 'nationality::Alerted Party Nationality (Normalized)', 'nationality::Hit Details (Nationality) (Normalized)', 'numeric::Matched Screening % (Parsed),alerted_party_age_years,alerted_party_dob_missing,hit_age_years,hit_dob_resolved,hit_dob_multi_value,hit_dob_unresolved,customer_prior_alert_count']


  PASS: no NaN/inf in final matrix



## 6. Unseen-category robustness (master plan section 19 requirement)

Fit on a slice, transform a slice containing a brand-new category value -- must not raise, must produce a zero-contribution row for that feature block rather than crashing scoring.

In [7]:
from features.entity_features import fit_entity_feature_artifacts, transform_entity_features

n = len(cv_df)
train_slice = cv_df.iloc[: n // 2]
holdout_slice = cv_df.iloc[n // 2 :].copy()
holdout_slice.iloc[0, holdout_slice.columns.get_loc("Branch Name")] = "A_BRANCH_NOT_IN_TRAINING"

train_artifacts = fit_entity_feature_artifacts(train_slice, "CustomerViolation")
holdout_matrix, _ = transform_entity_features(holdout_slice, "CustomerViolation", train_artifacts)
print(f"Transformed holdout with an unseen category, no error. Shape: {holdout_matrix.shape}")

Transformed holdout with an unseen category, no error. Shape: (1199, 5724)


## 7. Persist fitted artifacts (feature version, for reuse at inference)

In [8]:
from features.entity_features import save_entity_feature_artifacts

out_dir = REPO_ROOT / "models" / "entity"
manifests = {}
for sheet_name, artifacts in artifacts_by_sheet.items():
    save_entity_feature_artifacts(artifacts, out_dir)
    manifest_path = out_dir / f"{sheet_name}_{artifacts.feature_version}_manifest.json"
    manifests[sheet_name] = json.load(open(manifest_path))
    print(f"Saved {sheet_name} artifacts -> {manifest_path.name}")

print()
print(json.dumps(manifests["CustomerViolation"], indent=2, default=str))

Saved CustomerViolation artifacts -> CustomerViolation_entity-v1_manifest.json
Saved TransactionNameViolation artifacts -> TransactionNameViolation_entity-v1_manifest.json

{
  "sheet_name": "CustomerViolation",
  "feature_version": "entity-v1",
  "fitted_at": "2026-08-11T06:17:21.146627+00:00",
  "name_vectorizer_columns": [
    "Alerted Party Name (Normalized)",
    "Hit Details (Name) (Normalized)"
  ],
  "name_vocabulary_sizes": {
    "Alerted Party Name (Normalized)": 4296,
    "Hit Details (Name) (Normalized)": 4073
  },
  "categorical_columns": [
    "Customer Type",
    "Alerted Party",
    "Alert Type",
    "Sanctions Screening List Name",
    "Branch Name"
  ],
  "categorical_category_counts": {
    "Customer Type": 2,
    "Alerted Party": 2,
    "Alert Type": 3,
    "Sanctions Screening List Name": 11,
    "Branch Name": 41
  },
  "nationality_columns": [
    "Alerted Party Nationality (Normalized)",
    "Hit Details (Nationality) (Normalized)"
  ],
  "feature_block_order": 

## 8. Phase 3 feature-audit report

In [9]:
feature_audit = {
    "phase": "3_entity_features",
    "status": "PASS",
    "sheets": {
        sheet_name: {
            "rows": matrices[sheet_name].shape[0],
            "feature_dims": matrices[sheet_name].shape[1],
            "feature_blocks": artifacts_by_sheet[sheet_name].feature_names,
            "feature_version": artifacts_by_sheet[sheet_name].feature_version,
            "name_vocabulary_sizes": {
                col: len(vec.vocabulary_)
                for col, vec in artifacts_by_sheet[sheet_name].name_vectorizers.items()
            },
        }
        for sheet_name in entity_sheets
    },
    "checks": {
        "no_nan_or_inf_in_final_matrix": True,
        "no_leakage_column_used_as_feature_source": True,
        "unseen_category_handled_without_error": True,
        "historical_context_never_counts_future_alerts": True,
        "artifacts_persisted_for_reuse_at_inference": True,
    },
    "known_limitations": [
        "customer_prior_alert_count breaks ties among same-timestamp alerts for the "
        "same customer by original row order, not true causal order -- the source data "
        "has no finer-grained ordering signal for batch-generated same-second alerts.",
    ],
    "next_gate": "Phase 4 -- Transaction/Rule features. Requires human approval before proceeding.",
}

out_path = REPO_ROOT / "evaluation" / "phase3_entity_features_report.json"
with open(out_path, "w") as f:
    json.dump(feature_audit, f, indent=2, default=str)
print(f"Report written to {out_path}")
print(json.dumps(feature_audit, indent=2, default=str))

Report written to /home/chpl/Documents/AI-validation/AI_validator/Alert-AI/evaluation/phase3_entity_features_report.json
{
  "phase": "3_entity_features",
  "status": "PASS",
  "sheets": {
    "CustomerViolation": {
      "rows": 2397,
      "feature_dims": 8532,
      "feature_blocks": [
        "name_tfidf::Alerted Party Name (Normalized)",
        "name_tfidf::Hit Details (Name) (Normalized)",
        "categorical::Customer Type",
        "categorical::Alerted Party",
        "categorical::Alert Type",
        "categorical::Sanctions Screening List Name",
        "categorical::Branch Name",
        "nationality::Alerted Party Nationality (Normalized)",
        "nationality::Hit Details (Nationality) (Normalized)",
        "numeric::Matched Screening % (Parsed),alerted_party_age_years,alerted_party_dob_missing,hit_age_years,hit_dob_resolved,hit_dob_multi_value,hit_dob_unresolved,customer_prior_alert_count"
      ],
      "feature_version": "entity-v1",
      "name_vocabulary_sizes": 

## 9. Test suite

In [10]:
import subprocess

result = subprocess.run(["python", "-m", "pytest", "tests/", "-q"], cwd=REPO_ROOT, capture_output=True, text=True)
print(result.stdout[-2000:])
if result.returncode != 0:
    print(result.stderr[-2000:])
assert result.returncode == 0, "Test suite must pass before Phase 3 is considered done" 

........................................................................ [ 98%]
.                                                                        [100%]
73 passed in 4.96s



## Phase 3 -- Result

**Status: PASS**

- Character n-gram TF-IDF representation built for both name fields, both sheets -- the master-plan-recommended non-LLM baseline, no hand-written similarity threshold.
- DOB representation handles the Phase 2 finding (`Hit Details (DOB)` heterogeneity) via year-only fallback; unresolved cases flagged, not guessed.
- Nationality representation reuses Phase 2's normalized country values.
- Historical customer context is provably non-leaking (tested + demonstrated: strictly prior alerts only, non-decreasing over a customer's timeline). Known limitation: same-timestamp alerts for one customer are tie-broken by row order, not true causal order -- no finer signal exists in the source data.
- `Matched Screening %` included as a plain numeric feature, never a hardcoded threshold.
- Unseen-category and unseen-name robustness confirmed (required by master plan section 19).
- No leakage-typed column is ever a feature source (static guard at import + explicit test).
- Fitted artifacts persisted with a feature version tag and human-readable manifest for reuse at scoring time.
- 73/73 tests passing (12 new for this phase).

**Next gate:** Phase 4 -- Transaction/Rule feature representation (behavioural aggregates, rule context). Awaiting human approval to proceed.